In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import pandas as pd

# ── 경로 설정 ──────────────────────────────────────────
BASE = "/content/drive/MyDrive/[미래융합교육원] 1팀 공유사항/1차 머신러닝 프로젝트/@프로젝트 자료/03. 프로젝트 자료/data"

# ── 데이터 로드 ────────────────────────────────────────
# 사용 데이터:
# - results.csv    : 메인 학습 데이터 (1872~2026 국제경기 전체)
# - former_names.csv: 국가명 변경 이력 (국가명 통일용)
# - elo_ratings_wc2026.csv: 참고용 (실제 Elo는 eloratings.net 스크래핑으로 대체)
#
# 미사용 데이터 (로드 제외):
# - train.csv / test.csv : Kaggle 제공 WC 팀 레벨 데이터 → 우리 파이프라인과 맞지 않음
# - players.csv          : 빅5 리그 선수 스탯 → 연도별 데이터 없어 제외
# - shootouts.csv        : 승부차기 결과 → 시뮬레이션 단계에서 필요시 로드
# - goalscorers.csv      : 경기별 득점자 → 보조 분석용, 모델 학습엔 미사용

results = pd.read_csv(f"{BASE}/International football results from 1872 to 2026/results.csv")
former  = pd.read_csv(f"{BASE}/International football results from 1872 to 2026/former_names.csv")

# ── 로드 확인 ──────────────────────────────────────────
dfs = {"results": results, "former": former}
for name, df in dfs.items():
    print(f"{name:12s} | {df.shape[0]:>6,}행 × {df.shape[1]:>2}열")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/[미래융합교육원] 1팀 공유사항/1차 머신러닝 프로젝트/@프로젝트 자료/03. 프로젝트 자료/data/International football results from 1872 to 2026/results.csv'

In [ ]:
!pip install pycountry -q

In [ ]:
# ── STEP 1. 국가명 통일 ────────────────────────────────
# 모든 테이블이 팀명 문자열로 연결되므로 표기 불일치 시 조인 불가
# 파일마다 표기 다름 → 전처리 첫 단계로 통일 필수
#
# [주요 이슈]
# 1. former_names.csv: 국가명 변경 이력 36건 자동 처리
#    (예: Dahomey → Benin, Upper Volta → Burkina Faso)
#    컬럼명: 'former', 'current' (former_name/current_name 아님 주의)
# 2. 파일 간 표기 불일치 수동 보완
#    - South Korea vs Korea Republic
#    - Czech Republic vs Czechia
#    - IR Iran vs Iran
# 3. eloratings.net URL 예외 매핑은 STEP 3에서 처리

import pandas as pd

# 1-1. former_names 기반 + 수동 매핑 딕셔너리
name_map = dict(zip(former['former'], former['current']))
name_map.update({
    'South Korea':    'Korea Republic',
    'Czech Republic': 'Czechia',
    'Ivory Coast':    "Cote d'Ivoire",
    'IR Iran':        'Iran',
    'Curaçao':        'Curacao',
})

def normalize(name):
    if pd.isna(name): return name
    return name_map.get(name, name)

# 1-2. results 정규화
results['home_team'] = results['home_team'].map(normalize)
results['away_team'] = results['away_team'].map(normalize)

# 1-3. 48개국 기준 국가명 정합성 확인
wc_2026_teams = [
    'France', 'Spain', 'Argentina', 'England', 'Portugal', 'Brazil',
    'Netherlands', 'Morocco', 'Belgium', 'Germany', 'Croatia', 'Colombia',
    'Senegal', 'Mexico', 'United States', 'Uruguay', 'Japan', 'Switzerland',
    'Iran', 'Turkey', 'Ecuador', 'Austria', 'Korea Republic', 'Australia',
    'Algeria', 'Egypt', 'Canada', 'Norway', 'Panama', "Cote d'Ivoire",
    'Sweden', 'Paraguay', 'Czechia', 'Scotland', 'Tunisia', 'DR Congo',
    'Uzbekistan', 'Qatar', 'Iraq', 'South Africa', 'Saudi Arabia', 'Jordan',
    'Bosnia and Herzegovina', 'Cape Verde', 'Ghana', 'Curacao', 'Haiti',
    'New Zealand'
]

all_teams_in_results = set(results['home_team']) | set(results['away_team'])
missing = set(wc_2026_teams) - all_teams_in_results
print(f"48개국 중 results 누락: {missing if missing else '없음 ✅'}")
print(f"results: {len(results):,}행")

In [ ]:
# ── STEP 2. results.csv 뼈대 테이블 구성 ──────────────
# 모든 경기 데이터의 기준 테이블
# 나머지 데이터(Elo, 폼 등)는 전부 여기에 컬럼으로 붙는 구조
#
# [match_id]
# date + home_team + away_team + index 조합으로 경기 고유 식별자 생성
# → 같은 날 같은 팀끼리 두 경기 뛰는 경우(예: 1974 Tahiti vs New Caledonia)
#   정렬 기준 명확화 및 EWMA 폼 계산 시 중복 방지
# → EWMA 폼 계산, 파생변수 조인 등 이후 모든 단계에서 활용
#
# [데이터 품질 이슈]
# - Gibraltar vs Cayman Islands (2026-06-06): 동일 스코어 중복 입력 → 제거
# - Tahiti vs New Caledonia (1974-02-17): 스코어 다름 → 실제 두 경기 → 유지
#
# [레이블 정의]
# H = 홈팀 승 / D = 무승부 / A = 원정팀 승
# WC는 중립 경기(neutral=True)라 홈 이점 없음
# → 전체 H 비율(49%) vs WC H 비율(42%) 차이로 확인됨
#
# [관찰]
# 전체: H > A > D (홈 어드밴티지 뚜렷)
# WC:   H ≈ A    (중립 경기라 홈 이점 거의 없음)
# WC 무승부 비율 낮음(21%) → 연장/승부차기로 가는 경우 있음

results['date'] = pd.to_datetime(results['date'])
results['year'] = results['date'].dt.year

# 완전 중복 경기 제거 (날짜+팀+스코어 모두 동일한 경우 → 데이터 오류)
before = len(results)
results = results.drop_duplicates(
    subset=['date','home_team','away_team','home_score','away_score'],
    keep='first'
).reset_index(drop=True)
print(f"중복 제거: {before - len(results)}건 제거")

# match_id 생성 (index 포함 → 같은 날 같은 팀 두 경기도 구분 가능)
results['match_id'] = (results['date'].astype(str) + '_' +
                       results['home_team'] + '_' +
                       results['away_team'] + '_' +
                       results.index.astype(str))

def get_label(row):
    if row['home_score'] > row['away_score']:    return 'H'
    elif row['home_score'] == row['away_score']: return 'D'
    else:                                        return 'A'

results['result'] = results.apply(get_label, axis=1)

# WC 경기 별도 저장 (이변 분석, 우승팀 패턴 분석 등에 활용)
wc_results = results[results['tournament'] == 'FIFA World Cup'].copy()

# 확인
print(f"전체 경기: {len(results):,}건")
print(f"WC 경기:   {len(wc_results):,}건")
print(f"match_id 고유값: {results['match_id'].nunique():,}개")
print(f"\n레이블 분포 (전체):\n{results['result'].value_counts()}")
print(f"\n레이블 분포 (WC):\n{wc_results['result'].value_counts()}")

In [ ]:
# ── STEP 3-1. 스크래핑 준비 ───────────────────────────
# 기존 elo_ratings_wc2026.csv는 48개국만 커버
# → 상대팀 Elo 결측 33% 발생 → eloratings.net 직접 스크래핑으로 대체
#
# [TSV 구조]
# col0=년, col1=월, col2=일
# col3=홈팀코드, col4=원정팀코드
# col10=홈팀Elo, col11=원정팀Elo
#
# [주요 이슈]
# 1. 월/일이 0인 행 존재 (날짜 불명확) → merge_asof 조인 불가 → 제거
# 2. URL이 팀명과 다른 경우 수동 매핑 필요
#    - Korea Republic → South_Korea.tsv
#    - Cote d'Ivoire  → Ivory_Coast.tsv
#    - Republic of Ireland → Ireland.tsv
#    - São Tomé and Príncipe → Sao_Tome_and_Principe.tsv
#    - Réunion → Reunion.tsv
#    - Timor-Leste → TSV 없음 (HTML 반환) → 수집 불가
# 3. 해체국가(Yugoslavia, Czechoslovakia, German DR) / 지역팀(Guernsey 등)
#    → eloratings.net 미수록 → 결측 불가피 → 해당 경기 제거

import requests, time
from io import StringIO

# results.csv에 등장한 전체 팀 목록 추출
all_teams = set(results['home_team'].tolist()) | set(results['away_team'].tolist())

# eloratings.net URL이 팀명과 다른 경우 수동 매핑
url_override = {
    'Korea Republic':        'South_Korea',
    "Cote d'Ivoire":         'Ivory_Coast',
    'Republic of Ireland':   'Ireland',
    'São Tomé and Príncipe': 'Sao_Tome_and_Principe',
    'Réunion':               'Reunion',
}

print(f"전체 팀 수: {len(all_teams)}개")

In [ ]:
# ── STEP 3-2. 전체 팀 Elo 히스토리 스크래핑 ──────────
# 주의: 런타임당 1회만 실행 (약 2~3분 소요)
all_elo, failed = [], []

for team_name in sorted(all_teams):
    url_name = url_override.get(team_name, team_name.replace(' ', '_'))
    r = requests.get(f"https://www.eloratings.net/{url_name}.tsv")

    if r.status_code != 200:
        failed.append(team_name); continue

    try:
        df = pd.read_csv(StringIO(r.text), sep='\t', header=None)

        # TSV가 아닌 HTML 반환된 경우 (컬럼 수 12 미만) → 수집 불가
        if df.shape[1] < 12:
            failed.append(team_name); continue

        # 월/일=0인 행 제거 (날짜 불완전 → merge_asof 조인 불가)
        df = df[(df[1] != 0) & (df[2] != 0)]
        if len(df) == 0:
            failed.append(team_name); continue

        # 해당 팀 코드 = col3 최빈값 (홈경기 수 > 원정경기 수)
        team_code = df[3].value_counts().idxmax()

        # 홈(col3==code → col10) / 원정(col4==code → col11) Elo 추출
        home = df[df[3] == team_code][[0,1,2,10]].copy()
        away = df[df[4] == team_code][[0,1,2,11]].copy()
        for d in [home, away]: d.columns = ['year','month','day','rating']

        combined = pd.concat([home, away])
        combined['date'] = pd.to_datetime(combined[['year','month','day']], errors='coerce')
        combined = combined.dropna(subset=['date'])
        combined['country'] = team_name
        all_elo.append(combined[['date','country','rating']].sort_values('date').reset_index(drop=True))

    except Exception as e:
        failed.append(team_name); print(f"❌ {team_name} — {e}")

    time.sleep(0.2)

elo_full = pd.concat(all_elo).sort_values('date').reset_index(drop=True)
print(f"✅ {len(all_teams)-len(failed)}개국 완료 / ❌ {len(failed)}개국 실패")
print(f"총 {len(elo_full):,}건")

In [ ]:
# ── STEP 3-3. merge_asof로 Elo 조인 ──────────────────
# direction='backward': 경기 날짜 이전 가장 최근 Elo 사용
# 미래 Elo 사용 시 데이터 누출 발생 → 반드시 backward

# 48개국 누락 여부 먼저 체크 (치명적 누락 방지)
missing_48 = set(wc_2026_teams) - set(elo_full['country'].unique())
print(f"48개국 미수집: {missing_48 if missing_48 else '없음 ✅'}")

results = results.sort_values('date').reset_index(drop=True)

home_elo = pd.merge_asof(
    results[['date','home_team']],
    elo_full.rename(columns={'country':'home_team','rating':'home_elo'}),
    on='date', by='home_team', direction='backward'
)
away_elo = pd.merge_asof(
    results[['date','away_team']],
    elo_full.rename(columns={'country':'away_team','rating':'away_elo'}),
    on='date', by='away_team', direction='backward'
)

results['home_elo']  = home_elo['home_elo'].values
results['away_elo']  = away_elo['away_elo'].values
results['delta_elo'] = results['home_elo'] - results['away_elo']

In [ ]:
# ── STEP 3-4. 결측치 처리 ─────────────────────────────
# 수집 불가 케이스 → 해당 경기 제거
# - 해체국가(Yugoslavia, Czechoslovakia, German DR): eloratings.net 미수록
# - 지역팀(Guernsey, Basque Country 등): FIFA 비공인팀
# - Timor-Leste: TSV 미제공 (HTML 반환)
# → 평균 대체 시 노이즈만 추가됨 → 제거가 맞음

before = len(results)
results = results.dropna(subset=['home_elo','away_elo']).reset_index(drop=True)
after  = len(results)

print(f"제거 전: {before:,}건")
print(f"제거 후: {after:,}건 (제거: {before-after:,}건)")
print(f"\nElo 결측: {results['home_elo'].isna().sum()}건 ✅")
print(f"\n샘플 (한국 경기):")
print(results[
    (results['home_team']=='Korea Republic') |
    (results['away_team']=='Korea Republic')
][['date','home_team','away_team','home_elo','away_elo','delta_elo']].tail(5))

In [ ]:
# ── STEP 4. 단기 폼(EWMA) 계산 ────────────────────────
# 최근 경기에 더 높은 가중치를 준 이동평균 승점 → H2 가설 검증 피처
# shift(1) 필수: 현재 경기 결과가 폼에 포함되면 데이터 누출 발생
#
# [승점 기준]
# 홈팀: H=3, D=1, A=0 / 원정팀: H=0, D=1, A=3
#
# [EWMA 설정]
# span=5: 최근 5경기 기준 지수 가중 이동평균
# 최근 경기일수록 가중치 높음 → 단순 평균보다 최근 전력 변화 잘 반영
#
# [결측 처리]
# EWMA는 과거 경기 기반 지표 → 팀의 첫 경기에서는 계산 불가 → NaN 발생
# → missing 플래그 추가: 모델이 "첫 경기라 폼 데이터 없음"을 학습 가능
# → 폼 값은 1점(무승부 승점)으로 대체: 중립적인 승점 수준으로 간주
#
# [match_id 활용]
# 같은 날 같은 팀이 두 경기 뛰는 경우(222건 확인) → match_id로 정렬 기준 명확화
# → merge 시 match_id 기준으로 조인하여 중복 방지

# 4-1. 홈/원정 경기를 팀 기준 단일 테이블로 재구성
home_games = results[['date','match_id','home_team','result']].copy()
home_games['team']   = home_games['home_team']
home_games['points'] = home_games['result'].map({'H':3, 'D':1, 'A':0})

away_games = results[['date','match_id','away_team','result']].copy()
away_games['team']   = away_games['away_team']
away_games['points'] = away_games['result'].map({'H':0, 'D':1, 'A':3})

all_games = pd.concat([
    home_games[['date','match_id','team','points']],
    away_games[['date','match_id','team','points']]
]).sort_values(['team','date','match_id']).reset_index(drop=True)

# 4-2. EWMA 폼 계산
all_games['form_ewma'] = (
    all_games.groupby('team')['points']
    .transform(lambda x: x.shift(1).ewm(span=5, min_periods=1).mean())
)

# 4-3. results에 home_form / away_form 조인
# match_id 기준으로 조인 → 같은 날 두 경기 뛰는 경우도 정확히 매핑
home_form = all_games[['date','match_id','team','form_ewma']].rename(
    columns={'team':'home_team','form_ewma':'home_form'})
away_form = all_games[['date','match_id','team','form_ewma']].rename(
    columns={'team':'away_team','form_ewma':'away_form'})

results = results.merge(home_form, on=['date','match_id','home_team'], how='left')
results = results.merge(away_form, on=['date','match_id','away_team'], how='left')

# 4-4. 폼 결측 처리 (결측 여부 변수 보존)
# missing=1 → 모델이 "첫 경기라 폼 데이터 없음"을 학습 가능
# 1점(무승부 승점) 대체 → 중립적인 승점 수준으로 간주
results['home_form_missing'] = results['home_form'].isna().astype(int)
results['away_form_missing'] = results['away_form'].isna().astype(int)
results['home_form'] = results['home_form'].fillna(1)
results['away_form'] = results['away_form'].fillna(1)

# 확인
print(f"전체 경기 수: {len(results):,}건")
print(f"home_form 결측: {results['home_form'].isna().sum()}건")
print(f"away_form 결측: {results['away_form'].isna().sum()}건")
print(f"home_form_missing=1: {results['home_form_missing'].sum()}건")
print(f"away_form_missing=1: {results['away_form_missing'].sum()}건")
print(f"\n샘플 (한국 경기):")
print(results[
    (results['home_team']=='Korea Republic') |
    (results['away_team']=='Korea Republic')
][['date','home_team','away_team','home_form','away_form']].tail(5))

In [ ]:
# ── STEP 5. 대회 중요도 가중치 생성 (수정본) ──────────
# [수정 내용]
# 기존 코드에서 아래 예선 대회들이 매핑 안 됨 → NaN 발생 (6,036건)
#   - UEFA Euro qualification
#   - African Cup of Nations qualification
#   - AFC Asian Cup qualification
#   - CFU Caribbean Cup qualification
#   - CONCACAF Championship qualification
#   - Gold Cup qualification
#   - AFF Championship qualification
#   - COSAFA Cup qualification
#   - Arab Cup qualification
#   - Oceania Nations Cup qualification
#   - Copa América qualification
# → 기존 코드는 정확한 문자열 매칭만 하고 있었음
# → 수정: 'qualification' 포함 여부를 먼저 체크하는 방식으로 변경
#
# [가중치 체계]
# 1.00 : FIFA 월드컵 본선
# 0.85 : 대륙컵 본선 (유로, 코파아메리카, 아프리카컵, 아시안컵 등)
# 0.75 : 컨페더레이션스컵, 네이션스리그
# 0.70 : FIFA 월드컵 예선
# 0.60 : 주요 대륙컵 예선 (유로/코파/아프리카/아시안컵 예선)
# 0.50 : 기타 지역 예선 (CFU, COSAFA, AFF, Arab Cup 예선 등)
# 0.45 : 기타 지역대회 (머데카, 킹스컵 등 초청대회)
# 0.30 : 친선경기

def get_tournament_weight(tournament):
    t = str(tournament).lower()

    # 월드컵 본선
    if tournament == 'FIFA World Cup':
        return 1.0

    # 월드컵 예선 (본선 체크 이후)
    if 'fifa world cup qualification' in t:
        return 0.70

    # 대륙컵 예선 계열 — 먼저 체크 (본선보다 앞에 위치)
    if 'qualification' in t:
        # 주요 대륙 예선 (UEFA, CAF, AFC, CONMEBOL, CONCACAF 주요 대회)
        if any(x in t for x in [
            'uefa euro', 'copa am', 'african cup of nations',
            'afc asian cup', 'concacaf championship', 'gold cup',
        ]):
            return 0.60
        # 기타 지역 예선 (CFU, COSAFA, AFF, Arab Cup 등)
        else:
            return 0.50

    # 대륙컵 본선
    if any(x in t for x in [
        'uefa euro', 'copa am', 'african cup of nations', 'afc asian cup',
        'concacaf championship', 'gold cup', 'ofc nations cup',
        'oceania nations cup', 'aff championship', 'gulf cup', 'arab cup',
        'cecafa cup', 'cosafa cup', 'eaff championship',
        'cfu caribbean cup', 'uncaf cup',
    ]):
        return 0.85

    # 컨페더레이션스컵, 네이션스리그
    if any(x in t for x in [
        'confederations cup', 'nations league', 'concacaf nations'
    ]):
        return 0.75

    # 친선경기
    if 'friendly' in t:
        return 0.30

    # 나머지 기타 대회
    return 0.45

results['weight'] = results['tournament'].apply(get_tournament_weight)

print("=== 가중치 분포 ===")
print(results['weight'].value_counts().sort_index(ascending=False))
print(f"\nNaN 수: {results['weight'].isna().sum()}")

# 가중치별 대회 유형 확인
print("\n=== 가중치별 대표 대회 ===")
for w in sorted(results['weight'].unique(), reverse=True):
    sample = results[results['weight']==w]['tournament'].value_counts().head(3)
    print(f"\n  [{w}]")
    for t, c in sample.items():
        print(f"    {t}: {c:,}건")

In [ ]:
# ── STEP 6. 파생변수 생성 ─────────────────────────────
# results.csv 기반으로 경기 날짜 기준 동적 계산
#
# [피처 설계]
# 절대 전력 : 팀 자체 강도 지표
#   - home/away_form (STEP 4에서 생성)
#   - home/away_goal_diff_last_4y
#   - home/away_wc_participations, home/away_wc_titles
#   - home/away_win_rate_4y, home/away_goals_per_match_4y
# 상대 전력 : 홈/원정 팀 간 격차
#   - gap_form, gap_goal_diff, gap_wins
#   - gap_win_rate_4y, gap_goals_per_match_4y
#   - gap_wc_participations, gap_wc_titles
# 경기 수 보정 : matches_last_4y로 나눠 수치 왜곡 방지
#
# [홈/원정/중립 합산]
# 중립경기 표본 부족 → 쪼개면 노이즈 증가
# neutral 컬럼이 별도 피처로 존재 → 모델이 경기 성격 인식 가능
# → 전체 합산 기준 유지
#
# [결측 처리]
# - matches_last_4y=0 → 첫 경기 → 비율 계산 불가 → 0으로 대체
# - home/away_form_missing: EWMA는 matches와 독립적 → STEP 4에서 처리됨

import warnings
warnings.filterwarnings('ignore')

# 6-1. 팀 기준 득점/실점/승무패 테이블 구성
home_stats = results[['date','match_id','home_team','home_score','away_score','result']].copy()
home_stats.columns = ['date','match_id','team','scored','received','result']
home_stats['win']   = (home_stats['result'] == 'H').astype(int)
home_stats['draw']  = (home_stats['result'] == 'D').astype(int)
home_stats['loss']  = (home_stats['result'] == 'A').astype(int)
home_stats['match'] = 1

away_stats = results[['date','match_id','away_team','away_score','home_score','result']].copy()
away_stats.columns = ['date','match_id','team','scored','received','result']
away_stats['win']   = (away_stats['result'] == 'A').astype(int)
away_stats['draw']  = (away_stats['result'] == 'D').astype(int)
away_stats['loss']  = (away_stats['result'] == 'H').astype(int)
away_stats['match'] = 1

all_stats = pd.concat([home_stats, away_stats]).reset_index(drop=True)

# 6-2. 직전 4년 rolling 통계 계산
# sort_values(['date','match_id']): 같은 날 두 경기 순서 명확화
# shift(1): 현재 경기 결과 제외 (데이터 누출 방지)
# 1461일 = 365.25 * 4
def compute_rolling(group):
    group = group.sort_values(['date','match_id']).set_index('date')
    shifted = group[['scored','received','win','draw','loss','match']].shift(1)
    rolled  = shifted.rolling('1461D', min_periods=1).sum()
    group['goals_scored_last_4y']   = rolled['scored']
    group['goals_received_last_4y'] = rolled['received']
    group['wins_last_4y']           = rolled['win']
    group['draws_last_4y']          = rolled['draw']
    group['losses_last_4y']         = rolled['loss']
    group['matches_last_4y']        = rolled['match']
    return group.reset_index()

all_stats = all_stats.groupby('team', group_keys=False).apply(compute_rolling)

# 득실차 파생
all_stats['goal_diff_last_4y'] = (
    all_stats['goals_scored_last_4y'] - all_stats['goals_received_last_4y']
)

# 6-3. results에 홈/원정팀 기준으로 조인
stat_cols = [
    'goals_scored_last_4y', 'goals_received_last_4y',
    'wins_last_4y', 'draws_last_4y', 'losses_last_4y',
    'matches_last_4y', 'goal_diff_last_4y'
]

home_s = all_stats[['date','match_id','team'] + stat_cols].rename(columns={
    'team': 'home_team',
    **{col: f'home_{col}' for col in stat_cols}
})
away_s = all_stats[['date','match_id','team'] + stat_cols].rename(columns={
    'team': 'away_team',
    **{col: f'away_{col}' for col in stat_cols}
})

results = results.merge(home_s, on=['date','match_id','home_team'], how='left')
results = results.merge(away_s, on=['date','match_id','away_team'], how='left')

# 6-4. 결측 처리
# matches_last_4y=0이면 모델이 첫 경기임을 인식 가능
all_stat_cols = [f'home_{col}' for col in stat_cols] + [f'away_{col}' for col in stat_cols]
results[all_stat_cols] = results[all_stat_cols].fillna(0)

# 6-5. 경기 수 보정 피처 (matches_last_4y로 정규화)
# matches=0인 경우 0으로 대체 (첫 경기 → 비율 계산 불가)
for prefix in ['home', 'away']:
    m = results[f'{prefix}_matches_last_4y'].replace(0, pd.NA)
    results[f'{prefix}_win_rate_4y']         = (results[f'{prefix}_wins_last_4y'] / m).fillna(0)
    results[f'{prefix}_goals_per_match_4y']  = (results[f'{prefix}_goals_scored_last_4y'] / m).fillna(0)

# 6-6. WC 관련 파생변수
# 딕셔너리 기반으로 최적화 (apply 함수 내 딕셔너리 조회 → 속도 개선)
wc_appearances = wc_results[['date','home_team','away_team']].copy()
wc_appearances['year'] = wc_appearances['date'].dt.year

wc_years_home = wc_appearances.groupby('home_team')['year'].apply(set)
wc_years_away = wc_appearances.groupby('away_team')['year'].apply(set)
wc_years_dict = pd.concat([wc_years_home, wc_years_away]).groupby(level=0).apply(
    lambda x: set().union(*x)
).to_dict()

wc_winners = {
    1930:'Uruguay',   1934:'Italy',      1938:'Italy',
    1950:'Uruguay',   1954:'Germany',    1958:'Brazil',
    1962:'Brazil',    1966:'England',    1970:'Brazil',
    1974:'Germany',   1978:'Argentina',  1982:'Italy',
    1986:'Argentina', 1990:'Germany',    1994:'Brazil',
    1998:'France',    2002:'Brazil',     2006:'Italy',
    2010:'Spain',     2014:'Germany',    2018:'France',
    2022:'Argentina',
}

wc_titles_dict = {}
for year, winner in wc_winners.items():
    wc_titles_dict.setdefault(winner, set()).add(year)

def get_wc_participations(team, year):
    return sum(1 for y in wc_years_dict.get(team, set()) if y < year)

def get_wc_titles(team, year):
    return sum(1 for y in wc_titles_dict.get(team, set()) if y < year)

results['home_wc_participations'] = results.apply(
    lambda r: get_wc_participations(r['home_team'], r['date'].year), axis=1)
results['away_wc_participations'] = results.apply(
    lambda r: get_wc_participations(r['away_team'], r['date'].year), axis=1)
results['home_wc_titles'] = results.apply(
    lambda r: get_wc_titles(r['home_team'], r['date'].year), axis=1)
results['away_wc_titles'] = results.apply(
    lambda r: get_wc_titles(r['away_team'], r['date'].year), axis=1)

# 6-7. 대륙 매핑
continent_map = {
    # 유럽 (UEFA)
    'England':'UEFA', 'France':'UEFA', 'Germany':'UEFA', 'Spain':'UEFA',
    'Italy':'UEFA', 'Netherlands':'UEFA', 'Portugal':'UEFA', 'Belgium':'UEFA',
    'Croatia':'UEFA', 'Switzerland':'UEFA', 'Austria':'UEFA', 'Sweden':'UEFA',
    'Norway':'UEFA', 'Denmark':'UEFA', 'Scotland':'UEFA', 'Poland':'UEFA',
    'Czechia':'UEFA', 'Slovakia':'UEFA', 'Hungary':'UEFA', 'Romania':'UEFA',
    'Serbia':'UEFA', 'Bosnia and Herzegovina':'UEFA', 'Slovenia':'UEFA',
    'North Macedonia':'UEFA', 'Albania':'UEFA', 'Kosovo':'UEFA',
    'Montenegro':'UEFA', 'Bulgaria':'UEFA', 'Greece':'UEFA', 'Turkey':'UEFA',
    'Ukraine':'UEFA', 'Russia':'UEFA', 'Belarus':'UEFA', 'Moldova':'UEFA',
    'Estonia':'UEFA', 'Latvia':'UEFA', 'Lithuania':'UEFA', 'Finland':'UEFA',
    'Iceland':'UEFA', 'Ireland':'UEFA', 'Republic of Ireland':'UEFA',
    'Northern Ireland':'UEFA', 'Wales':'UEFA', 'Luxembourg':'UEFA',
    'Malta':'UEFA', 'Cyprus':'UEFA', 'Liechtenstein':'UEFA', 'Andorra':'UEFA',
    'San Marino':'UEFA', 'Gibraltar':'UEFA', 'Faroe Islands':'UEFA',
    'Armenia':'UEFA', 'Azerbaijan':'UEFA', 'Georgia':'UEFA',
    'Kazakhstan':'UEFA', 'Monaco':'UEFA', 'Northern Cyprus':'UEFA',
    'Israel':'UEFA',
    # 남미 (CONMEBOL)
    'Brazil':'CONMEBOL', 'Argentina':'CONMEBOL', 'Uruguay':'CONMEBOL',
    'Colombia':'CONMEBOL', 'Chile':'CONMEBOL', 'Paraguay':'CONMEBOL',
    'Ecuador':'CONMEBOL', 'Peru':'CONMEBOL', 'Bolivia':'CONMEBOL',
    'Venezuela':'CONMEBOL',
    # 북중미카리브 (CONCACAF)
    'United States':'CONCACAF', 'Mexico':'CONCACAF', 'Canada':'CONCACAF',
    'Costa Rica':'CONCACAF', 'Honduras':'CONCACAF', 'Panama':'CONCACAF',
    'Jamaica':'CONCACAF', 'El Salvador':'CONCACAF', 'Haiti':'CONCACAF',
    'Trinidad and Tobago':'CONCACAF', 'Curacao':'CONCACAF',
    'Guatemala':'CONCACAF', 'Cuba':'CONCACAF', 'Nicaragua':'CONCACAF',
    'Belize':'CONCACAF', 'Barbados':'CONCACAF', 'Dominican Republic':'CONCACAF',
    'Anguilla':'CONCACAF', 'Antigua and Barbuda':'CONCACAF',
    'Aruba':'CONCACAF', 'Bahamas':'CONCACAF', 'Bermuda':'CONCACAF',
    'Bonaire':'CONCACAF', 'British Virgin Islands':'CONCACAF',
    'Cayman Islands':'CONCACAF', 'Dominica':'CONCACAF',
    'French Guiana':'CONCACAF', 'Grenada':'CONCACAF',
    'Guadeloupe':'CONCACAF', 'Guyana':'CONCACAF',
    'Martinique':'CONCACAF', 'Montserrat':'CONCACAF',
    'Puerto Rico':'CONCACAF', 'Saint Kitts and Nevis':'CONCACAF',
    'Saint Lucia':'CONCACAF', 'Saint Martin':'CONCACAF',
    'Saint Pierre and Miquelon':'CONCACAF', 'Sint Maarten':'CONCACAF',
    'Saint Vincent and the Grenadines':'CONCACAF',
    'Suriname':'CONCACAF', 'Turks and Caicos Islands':'CONCACAF',
    # 아프리카 (CAF)
    'Morocco':'CAF', 'Senegal':'CAF', 'Nigeria':'CAF', 'Ghana':'CAF',
    'Cameroon':'CAF', 'Algeria':'CAF', 'Tunisia':'CAF', 'Egypt':'CAF',
    "Cote d'Ivoire":'CAF', 'DR Congo':'CAF', 'Mali':'CAF', 'Burkina Faso':'CAF',
    'Cape Verde':'CAF', 'South Africa':'CAF', 'Zimbabwe':'CAF', 'Zambia':'CAF',
    'Tanzania':'CAF', 'Kenya':'CAF', 'Ethiopia':'CAF', 'Angola':'CAF',
    'Mozambique':'CAF', 'Uganda':'CAF', 'Rwanda':'CAF', 'Benin':'CAF',
    'Gabon':'CAF', 'Congo':'CAF', 'Guinea':'CAF', 'Guinea-Bissau':'CAF',
    'Equatorial Guinea':'CAF', 'Togo':'CAF', 'Niger':'CAF', 'Sudan':'CAF',
    'Libya':'CAF', 'Somalia':'CAF', 'Comoros':'CAF', 'Madagascar':'CAF',
    'Mauritania':'CAF', 'Gambia':'CAF', 'Sierra Leone':'CAF', 'Liberia':'CAF',
    'Central African Republic':'CAF', 'Chad':'CAF', 'Burundi':'CAF',
    'Botswana':'CAF', 'Chagos Islands':'CAF', 'Djibouti':'CAF',
    'Eritrea':'CAF', 'Eswatini':'CAF', 'Lesotho':'CAF', 'Malawi':'CAF',
    'Mauritius':'CAF', 'Mayotte':'CAF', 'Namibia':'CAF', 'Réunion':'CAF',
    'Seychelles':'CAF', 'Somaliland':'CAF', 'South Sudan':'CAF',
    'São Tomé and Príncipe':'CAF', 'Western Sahara':'CAF', 'Zanzibar':'CAF',
    # 아시아 (AFC)
    'Korea Republic':'AFC', 'Japan':'AFC', 'Iran':'AFC', 'Saudi Arabia':'AFC',
    'Australia':'AFC', 'Qatar':'AFC', 'Iraq':'AFC', 'Jordan':'AFC',
    'Uzbekistan':'AFC', 'China':'AFC', 'India':'AFC', 'Thailand':'AFC',
    'Vietnam':'AFC', 'Indonesia':'AFC', 'Malaysia':'AFC',
    'United Arab Emirates':'AFC', 'Oman':'AFC', 'Bahrain':'AFC',
    'Kuwait':'AFC', 'Syria':'AFC', 'Lebanon':'AFC', 'North Korea':'AFC',
    'Taiwan':'AFC', 'Hong Kong':'AFC', 'Singapore':'AFC', 'Myanmar':'AFC',
    'Cambodia':'AFC', 'Tajikistan':'AFC', 'Kyrgyzstan':'AFC',
    'Turkmenistan':'AFC', 'Mongolia':'AFC', 'Bangladesh':'AFC', 'Nepal':'AFC',
    'Sri Lanka':'AFC', 'Pakistan':'AFC', 'Afghanistan':'AFC', 'Maldives':'AFC',
    'Bhutan':'AFC', 'Laos':'AFC', 'Philippines':'AFC', 'Brunei':'AFC',
    'Guam':'AFC', 'Macau':'AFC', 'Chinese Taipei':'AFC',
    'Kurdistan':'AFC', 'North Vietnam':'AFC', 'Palestine':'AFC',
    'Tibet':'AFC', 'Yemen':'AFC',
    # 오세아니아 (OFC)
    'New Zealand':'OFC', 'Fiji':'OFC', 'Papua New Guinea':'OFC',
    'Vanuatu':'OFC', 'Solomon Islands':'OFC', 'Samoa':'OFC',
    'Tonga':'OFC', 'Cook Islands':'OFC', 'Tahiti':'OFC',
    'New Caledonia':'OFC', 'Falkland Islands':'OFC', 'Greenland':'OFC',
    'Kiribati':'OFC', 'Marshall Islands':'OFC', 'Niue':'OFC',
    'Northern Mariana Islands':'OFC', 'Palau':'OFC', 'Tuvalu':'OFC',
}

results['home_continent'] = results['home_team'].map(continent_map)
results['away_continent'] = results['away_team'].map(continent_map)

missing_continent = set(
    results[results['home_continent'].isna()]['home_team'].tolist() +
    results[results['away_continent'].isna()]['away_team'].tolist()
)
print(f"대륙 매핑 안 된 팀 수: {len(missing_continent)}")

# 6-8. 개최국 여부
# 공동개최의 경우 모든 개최국 포함
host_map = {
    1930: ['Uruguay'],
    1934: ['Italy'],        1938: ['France'],
    1950: ['Brazil'],       1954: ['Switzerland'],
    1958: ['Sweden'],       1962: ['Chile'],
    1966: ['England'],      1970: ['Mexico'],
    1974: ['Germany'],      1978: ['Argentina'],
    1982: ['Spain'],        1986: ['Mexico'],
    1990: ['Italy'],        1994: ['United States'],
    1998: ['France'],       2002: ['Korea Republic', 'Japan'],
    2006: ['Germany'],      2010: ['South Africa'],
    2014: ['Brazil'],       2018: ['Russia'],
    2022: ['Qatar'],        2026: ['United States', 'Canada', 'Mexico'],
}

def get_is_host(team, year):
    return 1 if team in host_map.get(year, []) else 0

results['home_is_host'] = results.apply(
    lambda r: get_is_host(r['home_team'], r['date'].year), axis=1)
results['away_is_host'] = results.apply(
    lambda r: get_is_host(r['away_team'], r['date'].year), axis=1)

# 6-9. gap 변수 (상대 전력 격차)
# wc 관련 gap은 wc 계산 이후에 생성
results['gap_form']               = results['home_form']               - results['away_form']
results['gap_goal_diff']          = results['home_goal_diff_last_4y']  - results['away_goal_diff_last_4y']
results['gap_wins']               = results['home_wins_last_4y']       - results['away_wins_last_4y']
results['gap_win_rate_4y']        = results['home_win_rate_4y']        - results['away_win_rate_4y']
results['gap_goals_per_match_4y'] = results['home_goals_per_match_4y'] - results['away_goals_per_match_4y']
results['gap_wc_participations']  = results['home_wc_participations']  - results['away_wc_participations']
results['gap_wc_titles']          = results['home_wc_titles']          - results['away_wc_titles']
results['is_wc'] = (results['tournament'] == 'FIFA World Cup').astype(int)

# 최종 확인
print(f"전체 경기 수: {len(results):,}건")
print(f"컬럼 수: {len(results.columns)}개")
print(f"\n샘플 (한국 경기):")
print(results[
    (results['home_team']=='Korea Republic') |
    (results['away_team']=='Korea Republic')
][['date','home_team','away_team',
   'home_win_rate_4y','home_goals_per_match_4y',
   'gap_form','gap_wc_participations','gap_wc_titles']].tail(5).to_string())

6-9 gap 변수 바로 아래(6-10. 예선 난이도)로 추가
이유
1. continent_map이 이미 정의된 후라 바로 참조 가능
2. gap 변수들과 같은 성격 (상대 전력 격차 계산)
3. FEATURES 리스트 확정 전 (STEP 7) 이므로 자연스럽게 포함됨

이점
1. "예선 강도"를 통제변수로 씀
2. SHAP 해석에서 대륙 원핫보다 유리
3. WC 연도별로 슬롯이 달라지는 걸 반영

STEP 7 FEATURES 리스트에는 이렇게 추가
-> # gap (상대 전력) 섹션에 함께 추가
'home_qual_rate', 'away_qual_rate', 'gap_qual_rate',

단, 1930~1994 경기는 결측이 생기니까 STEP 7 직전에 fillna(0.5) 한 줄 넣어두는 게 안전
for col in ['home_qual_rate', 'away_qual_rate', 'gap_qual_rate']:
    results[col] = results[col].fillna(0.5)

In [ ]:
# # ── 대륙별 WC 예선 통과율 (WC 연도별) ─────────────────
# # 직행 슬롯 / 예선 참가국 수 (호스트 자동진출은 제외하고 계산)
# # 출처: 각 WC 위키피디아 예선 페이지

# wc_qualification_rate = {
#     # (UEFA, CAF, AFC, CONCACAF, CONMEBOL, OFC)
#     1930: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),  # 초청제
#     1934: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1938: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1950: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1954: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1958: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1962: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1966: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1970: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1974: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1978: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1982: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1986: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1990: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     1994: dict(UEFA=None, CAF=None, AFC=None, CONCACAF=None, CONMEBOL=None, OFC=None),
#     # 32팀 시대 (1998~2022) — 슬롯/참가국 수 실측값
#     1998: dict(UEFA=15/49, CAF=5/38, AFC=4/39,  CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2002: dict(UEFA=15/50, CAF=5/51, AFC=4.5/39, CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2006: dict(UEFA=14/51, CAF=5/51, AFC=4.5/40, CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2010: dict(UEFA=13/53, CAF=6/52, AFC=4.5/43, CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2014: dict(UEFA=13/53, CAF=5/54, AFC=4.5/43, CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2018: dict(UEFA=14/54, CAF=5/54, AFC=4.5/46, CONCACAF=3/30, CONMEBOL=4.5/10, OFC=0.5/11),
#     2022: dict(UEFA=13/54, CAF=5/54, AFC=4/46,   CONCACAF=3/30, CONMEBOL=4/10,   OFC=0.5/11),
#     # 48팀 시대
#     2026: dict(UEFA=16/54, CAF=9/53, AFC=8/46, CONCACAF=6/32, CONMEBOL=6/10, OFC=1/11),
# }

# def get_qual_rate(team, match_year, continent_map, wc_qualification_rate):
#     continent = continent_map.get(team)
#     if continent is None:
#         return None

#     # 해당 팀이 속한 WC 연도 찾기 (경기 연도 이전 가장 최근 WC 기준)
#     wc_years = sorted(wc_qualification_rate.keys())
#     target_wc = None
#     for y in wc_years:
#         if y <= match_year:
#             target_wc = y

#     if target_wc is None:
#         return None

#     rate_dict = wc_qualification_rate[target_wc]
#     return rate_dict.get(continent)  # 1930~1994는 None 반환

# # results에 적용
# results['home_qual_rate'] = results.apply(
#     lambda r: get_qual_rate(r['home_team'], r['year'], continent_map, wc_qualification_rate), axis=1)
# results['away_qual_rate'] = results.apply(
#     lambda r: get_qual_rate(r['away_team'], r['year'], continent_map, wc_qualification_rate), axis=1)
# results['gap_qual_rate'] = results['home_qual_rate'] - results['away_qual_rate']

# # FEATURES에 추가
# FEATURES += ['home_qual_rate', 'away_qual_rate', 'gap_qual_rate']

# # 결측 확인 (1930~1994 경기들은 None)
# print(f"home_qual_rate 결측: {results['home_qual_rate'].isna().sum():,}건")
# print(f"결측 기간 경기 비율: {results['home_qual_rate'].isna().mean():.1%}")

In [ ]:
# 최종 컬럼 목록 확인
print(f"전체 경기 수: {len(results):,}건")
print(f"\n컬럼 목록:")
for col in results.columns.tolist():
    print(f"  - {col}")

In [ ]:
# ── STEP 7. 최종 피처 테이블 구성 ─────────────────────
# 모델에 입력할 피처 확정 및 레이블 생성
#
# [제외 컬럼]
# - date, home_team, away_team, match_id : 식별자
# - home_score, away_score               : 경기 후 확인 가능 → 데이터 누출
# - tournament, city, country, year      : 직접 피처로 안 씀
# - result                               : y로 대체
# - weight                               : sample_weight로 별도 관리
#
# [레이블 인코딩]
# H=2 (홈승) / D=1 (무) / A=0 (원정승)
#
# [대륙 원핫인코딩]
# 문자열 → 모델 입력 불가 → pd.get_dummies로 0/1 변환
#
# [train/test split]
# random split 절대 금지 → 시간 기준 split 필수
# split 기준: 2022-01-01 (2022 WC 이전/이후)
# → 모델 성능 평가용 (2026 WC 예측은 별도 시뮬레이션 단계)

# 7-1. 레이블 생성
results['y'] = results['result'].map({'H':2, 'D':1, 'A':0})

# 7-2. 대륙 원핫인코딩
results = pd.get_dummies(results, columns=['home_continent','away_continent'])

# 7-3. 피처 목록 확정
continent_cols = [c for c in results.columns if 'continent' in c]

FEATURES = [
    # Elo
    'delta_elo', 'home_elo', 'away_elo',
    # 단기 폼
    'home_form', 'away_form',
    'home_form_missing', 'away_form_missing',
    # 경기 환경
    'neutral', 'is_wc',
    # 직전 4년 성적 (절대)
    'home_goals_scored_last_4y', 'home_goals_received_last_4y',
    'home_wins_last_4y', 'home_draws_last_4y', 'home_losses_last_4y',
    'home_matches_last_4y', 'home_goal_diff_last_4y',
    'away_goals_scored_last_4y', 'away_goals_received_last_4y',
    'away_wins_last_4y', 'away_draws_last_4y', 'away_losses_last_4y',
    'away_matches_last_4y', 'away_goal_diff_last_4y',
    # 경기 수 보정
    'home_win_rate_4y', 'home_goals_per_match_4y',
    'away_win_rate_4y', 'away_goals_per_match_4y',
    # WC 경험
    'home_wc_participations', 'away_wc_participations',
    'home_wc_titles', 'away_wc_titles',
    # 개최국
    'home_is_host', 'away_is_host',
    # gap (상대 전력)
    'gap_form', 'gap_goal_diff', 'gap_wins',
    'gap_win_rate_4y', 'gap_goals_per_match_4y',
    'gap_wc_participations', 'gap_wc_titles',
] + continent_cols

# 7-4. 날짜 기준 split
split_date = '2022-01-01'
train_df = results[results['date'] <  split_date]
test_df  = results[results['date'] >= split_date]

X_train = train_df[FEATURES]
y_train = train_df['y']
X_test  = test_df[FEATURES]
y_test  = test_df['y']
w_train = train_df['weight']

# 확인
print(f"학습: {len(X_train):,}행 / 테스트: {len(X_test):,}행")
print(f"피처 수: {len(FEATURES)}개")
print(f"\n레이블 분포 (학습):\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\n레이블 분포 (테스트):\n{y_test.value_counts(normalize=True).round(3)}")

In [ ]:
# ── 전처리 완료 데이터 저장 ────────────────────────────
# 런타임 재시작 시 전처리 재실행 방지
# Google Drive에 저장 → 영구 보관

SAVE_PATH = f"{BASE}/preprocessed"
import os
os.makedirs(SAVE_PATH, exist_ok=True)

# 전체 results 저장
results.to_csv(f"{SAVE_PATH}/results_preprocessed.csv", index=False)

# train/test split 저장
X_train.to_csv(f"{SAVE_PATH}/X_train.csv", index=False)
X_test.to_csv(f"{SAVE_PATH}/X_test.csv", index=False)
y_train.to_csv(f"{SAVE_PATH}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_PATH}/y_test.csv", index=False)
w_train.to_csv(f"{SAVE_PATH}/w_train.csv", index=False)

# FEATURES 목록 저장 (나중에 모델링 단계에서 불러올 때 필요)
import json
with open(f"{SAVE_PATH}/features.json", 'w') as f:
    json.dump(FEATURES, f)

print(f"저장 완료: {SAVE_PATH}")
print(f"  - results_preprocessed.csv : {len(results):,}행 × {len(results.columns)}열")
print(f"  - X_train.csv              : {len(X_train):,}행 × {len(X_train.columns)}열")
print(f"  - X_test.csv               : {len(X_test):,}행 × {len(X_test.columns)}열")
print(f"  - y_train.csv              : {len(y_train):,}행")
print(f"  - y_test.csv               : {len(y_test):,}행")
print(f"  - w_train.csv              : {len(w_train):,}행")
print(f"  - features.json            : {len(FEATURES)}개 피처")

In [ ]:
# ── 최종 결측치 확인 ──────────────────────────────────
print("=" * 50)
print("최종 결측치 확인")
print("=" * 50)

for name, data in [
    ("X_train", X_train),
    ("X_test",  X_test),
    ("y_train", y_train),
    ("y_test",  y_test),
    ("w_train", w_train),
]:
    nan_count = data.isna().sum().sum() if hasattr(data, 'columns') else data.isna().sum()
    status    = "✅" if nan_count == 0 else "❌"
    print(f"  {status} {name:10s}: NaN {nan_count}개")

print(f"\n✅ 전처리 완료")
print(f"   X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"   피처 수: {len(FEATURES)}개")

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print(df.shape)
print(df.head(3))